<a href="https://colab.research.google.com/github/AlejandroAC977/ETL-pySpark/blob/main/ETLSpark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**ETL Con PySpark**

In [ ]:
import pyspark

from pyspark.sql import SparkSession, SQLContext
from pyspark.sql.types import IntegerType

spark = SparkSession.builder.appName('app').getOrCreate()
sqlContext = SQLContext(spark)


/usr/local/lib/python3.12/dist-packages/pyspark/sql/context.py:112: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


Carga de Datos a travez de Drive

In [ ]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

In [ ]:
path = "/content/drive/MyDrive/datasets/"

Lectura con una Definicion del esquema de tipo de datos ay qeu el csv cuenta con los tipos en la columna 1

In [ ]:
schema = StructType([
    StructField("Car", StringType(), True),
    StructField("MPG", DoubleType(), True),
    StructField("Cylinders", IntegerType(), True),
    StructField("Displacement", DoubleType(), True),
    StructField("Horsepower", DoubleType(), True),
    StructField("Weight", DoubleType(), True),
    StructField("Acceleration", DoubleType(), True),
    StructField("Model", IntegerType(), True),
    StructField("Origin", StringType(), True)
])
df = spark.read.csv(path+"cars.csv", header=True, sep=";", schema=schema)

Visualizar datos

In [ ]:
display(df)

DataFrame[Car: string, MPG: double, Cylinders: int, Displacement: double, Horsepower: double, Weight: double, Acceleration: double, Model: int, Origin: string]

In [ ]:
df.show()

In [ ]:
df.select('car', 'cylinders', 'Horsepower', 'weight')

DataFrame[car: string, cylinders: int, Horsepower: double, weight: double]

RDD

In [ ]:
def dropFirst(index, i):
  return iter(list(i)[1:])

In [ ]:
carsRDD = df.rdd

In [ ]:
carsRDD.collect()

Al ser una unica particion podemos eliminar la primera fila sin problemas

In [ ]:
carsRDD = carsRDD.mapPartitionsWithIndex(dropFirst)

In [ ]:
carsRDD.collect()

Creamos un df apartir de nuestro rdd

In [ ]:
dataf = sqlContext.createDataFrame(carsRDD, schema)
dataf.printSchema()

In [ ]:
dataf.show()

Editar tipo de dato en columna

In [ ]:
dataf = dataf.withColumn("Displacement", dataf["Displacement"].cast(IntegerType()))

In [ ]:
dataf.printSchema()

Creacion de Columna Calculada

In [ ]:
dataf = dataf.withColumn("Acel/Peso", dataf["Weight"]/dataf["Acceleration"])

In [ ]:
display(dataf)
dataf.printSchema()
dataf.show()

**Particiones**

In [ ]:
dataf.repartition(1).write.csv('particion', sep= ";")

In [ ]:
dataf.cache()
dataf.count() # Force the cache to be materialized
parquet_output_path = path + "parquet_cars"
dataf.write.mode("overwrite").parquet(parquet_output_path)